# 14_train_evaluate_supervised_ML_models

Run supervised ML training/evaluation across feature combinations, split types, and model choices.

## 1) Environment Setup


In [1]:
from pathlib import Path
import sys

# Section 1: Resolve repo root from either project root or notebooks/ cwd.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

# Section 2: Add repo and src roots to sys.path for imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


repo_root: /Users/charmainechia/Documents/projects/AbCode
src_root: /Users/charmainechia/Documents/projects/AbCode/src


## 2) Imports


In [3]:
from pprint import pprint
from project_config.feature_registry import COMBI_ML_FEATURE_SETS
from project_config.variables import address_dict


## 3) Configure ML Inputs


In [4]:
import abcode.steps.train_evaluate_supervised_ml_models as ml_step
default_user_inputs = ml_step.default_user_inputs
user_inputs = default_user_inputs()

# Edit these for your run
user_inputs['root_key'] = 'examples' # 'biostream-developability-data'
user_inputs['data_fbase'] = user_inputs['root_key']
user_inputs['data_subfolder'] = 'opensource' #
user_inputs['csv_suffix'] = '_asset_acsins_xgboost' #
default_dataset_fname = user_inputs['data_subfolder'] + '.csv'
user_inputs['dataset_fname'] = default_dataset_fname

# Experimental data parent folder: address_dict[root_key] / "expdata" / data_subfolder
expdata_dir = (repo_root / Path(address_dict[user_inputs['root_key']]) / 'expdata' / user_inputs['data_subfolder']).resolve()
user_inputs['input_filename_prefix'] = user_inputs['data_subfolder'] + '_' # ''
user_inputs['sequence_base'] = None
user_inputs['target_col'] = ['acsins_dLmax_ph7.4_avg'] # ['hic_rt_avg', 'sec_%monomer_avg', 'acsins_dLmax_ph7.4_avg', 'tm1_nanodsf_avg']
user_inputs['classification_or_regression'] = 'regression'

# specify splits
# options include: 'random', 'mutres-modulo', 'contiguous', 'asset', 'custom'
user_inputs['split_type_list'] = ['asset'] # ['random']
user_inputs['k_folds'] = 5
user_inputs['random_kfold_repeats'] = 1  # >1 enables repeated random k-fold from scratch
user_inputs['random_split_col'] = f'fold_random_{user_inputs["k_folds"]}'
user_inputs['mutres_split_col'] = f'fold_mutres-modulo_{user_inputs["k_folds"]}'
user_inputs['contiguous_split_col'] = f'fold_contiguous_{user_inputs["k_folds"]}'  # optional precomputed column
user_inputs['asset_split_col'] = 'dataset'  # used when split_type_list includes 'asset'
user_inputs['custom_split_col'] = 'fold_custom'
user_inputs['custom_test_value'] = 1
# specify custom dataset name
user_inputs['custom_test_dataset_fname'] = None #
user_inputs['custom_test_data_subfolder'] = user_inputs['data_subfolder']
user_inputs['custom_input_filename_prefix'] = None if user_inputs['custom_test_dataset_fname'] is None else user_inputs['custom_test_dataset_fname'].split('.')[0] # ''

# specify featureset combinations to test
BEST_FEATURES = {
        'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    'physicochemical_seq_all': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac'],
}
# user_inputs['feature_combinations_dict'] = COMBI_ML_FEATURE_SETS
user_inputs['feature_combinations_dict'] = {
    # 'vh_esm2_seq_embeddings': ['vh_esm2-650m_mean_pooled-33'],
    # 'cdrh3_esm2_seq_embeddings': ['cdrh3_esm2-650m_mean_pooled-33'],
    # 'vh_cdrh3_esm2_seq_embeddings': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_mean_pooled-33'],
    # 'vh_esm2_seq_embeddings_cdrh_esm2_pll': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_PLL-wt', ],
    # 'vh_esm2_seq_embeddings_svd20_cdrh_esm2_pll': ['vh_esm2-650m_mean_pooled-33_svd20', 'cdrh3_esm2-650m_PLL-wt', ],
    # 'vh_vl_georgiev_mean_pooled': ['vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled']
    # 'vh_esm2_seq_embeddings_cdrh_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'vh_esm2_seq_embeddings_svd20_cdrh_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd20', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'vhvlcdrh3_esm2_seq_embeddings_svd50_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd50', 'vl_esm2-650m_mean_pooled-33_svd50', 'cdrh3_esm2-650m_mean_pooled-33_svd50', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    #     'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'physicochemical_seq_all': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac'],
    # 'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
        'physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac','vh_esm2-650m_mean_pooled-33_svd50', 'vl_esm2-650m_mean_pooled-33_svd50', 'cdrh3_esm2-650m_mean_pooled-33_svd50', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt','vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
}


# specify models
user_inputs['model_list'] =  ['xgboost']# ['ridge', 'mlp_pytorch'] # ['ridge']

# run parameters
# hyperparameter tuning
user_inputs['hyperparameter_mode'] = 'optuna_search' # None # 'preset_search' # 'default', 'ntrain_preset',
user_inputs['tuning_metric'] = 'spearman'
user_inputs['tuning_n_trials'] = 20
user_inputs['summary_source_mode'] = 'all_saved_rows'  # 'run_cache' or 'all_saved_rows'
user_inputs['summary_metric_mode'] = 'average'  # 'average' or 'pooled'

# save parameters
user_inputs['save_trained_models'] = False
user_inputs['save_predictions'] = True
user_inputs['train_full_data_model'] = False
user_inputs['featurecombi_model_pair_to_extract_coefficients_for'] = None # [('onehot', 'ridge')]
user_inputs['show_progress'] = True
pprint(user_inputs)


{'asset_split_col': 'dataset',
 'classification_or_regression': 'regression',
 'contiguous_split_col': 'fold_contiguous_5',
 'csv_suffix': '_asset_acsins_xgboost',
 'custom_input_filename_prefix': None,
 'custom_split_col': 'fold_custom',
 'custom_test_data_subfolder': 'opensource',
 'custom_test_dataset_fname': None,
 'custom_test_value': 1,
 'data_fbase': 'examples',
 'data_subfolder': 'opensource',
 'dataset_fname': 'opensource.csv',
 'feature_combinations_dict': {'physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev': ['cdr_length',
                                                                                              'vh_aac',
                                                                                              'vh_aaindex1',
                                                                                              'vh_ctdc',
                                                                                              'vh_ctdc',
                            

## 4) Run Training And Evaluation


In [5]:
import importlib

import abcode.tools.ml.model_registry as model_registry
import abcode.tools.ml.hyperparameter_tuning as hyperparameter_tuning
import abcode.tools.ml.workflow as workflow
import abcode.steps.train_evaluate_supervised_ml_models as ml_step

importlib.reload(model_registry)
importlib.reload(hyperparameter_tuning)
importlib.reload(workflow)
ml_step = importlib.reload(ml_step)

result = ml_step.train_evaluate_supervised_ml_models(user_inputs)
pprint(result)


[I 2026-04-25 15:55:40,754] A new study created in memory with name: no-name-fe09f623-2289-4eb7-9898-2db8c6d688c6


[row-filter] context=target_col=acsins_dLmax_ph7.4_avg | split_type=asset | feature_combi_name=physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev | dropped_rows=150 | remaining_rows=517
[eval-start] target_col=acsins_dLmax_ph7.4_avg | split_type=asset | feature_combi_name=physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev | model_name=xgboost


[I 2026-04-25 15:55:57,569] Trial 0 finished with value: 0.4201008242956652 and parameters: {'learning_rate': 0.023688639503640783, 'max_depth': 6, 'n_estimators': 500}. Best is trial 0 with value: 0.4201008242956652.


[tuning-trial] model=xgboost | metric=spearman | trial=1/20 | value=0.420101 | best=0.420101


[I 2026-04-25 15:56:00,183] Trial 1 finished with value: 0.1708994845935962 and parameters: {'learning_rate': 0.03968793330444373, 'max_depth': 3, 'n_estimators': 300}. Best is trial 0 with value: 0.4201008242956652.


...

[tuning-trial] model=xgboost | metric=spearman | trial=18/20 | value=0.358766 | best=0.454071


[I 2026-04-25 15:59:34,098] Trial 18 finished with value: 0.3339290731919655 and parameters: {'learning_rate': 0.018443042488457084, 'max_depth': 5, 'n_estimators': 600}. Best is trial 16 with value: 0.454071338660383.


[tuning-trial] model=xgboost | metric=spearman | trial=19/20 | value=0.333929 | best=0.454071


[I 2026-04-25 15:59:47,072] Trial 19 finished with value: 0.4123392296214735 and parameters: {'learning_rate': 0.04121734223967981, 'max_depth': 6, 'n_estimators': 400}. Best is trial 16 with value: 0.454071338660383.


[tuning-trial] model=xgboost | metric=spearman | trial=20/20 | value=0.412339 | best=0.454071
[tuning-complete] model=xgboost | metric=spearman | direction=maximize | best_value=0.454071 | n_trials=20
[tuning-selected] model=xgboost | selected_hyperparameters={'n_estimators': 600, 'learning_rate': 0.0174427425592466, 'max_depth': 6, 'random_state': 42}
[fold-result] split_id=0 | split_name=asset_GDPa1 | split_type=asset | test_asset=GDPa1 | eval_group=all_data | data_size_n=517 | n_train=273 | n_test=244
 test_spearman  test_pearson  test_r2  test_rmse  train_spearman  train_pearson  train_r2  train_rmse
          0.67        0.6475   0.3321     6.9585          0.9992            1.0    0.9999      0.0712
[eval-start] target_col=acsins_dLmax_ph7.4_avg | split_type=asset | feature_combi_name=physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev | model_name=xgboost
[fold-result] split_id=1 | split_name=asset_GDPa3 | split_type=asset | test_asset=GDPa3 | eval_group=all_data | data_si

## 6) Visualize ML Results


In [ ]:
from pprint import pprint

import importlib

import abcode.steps.visualize_ml_results as ml_viz_step
ml_viz_step = importlib.reload(ml_viz_step)

visualize_ml_results = ml_viz_step.visualize_ml_results
default_viz_inputs = ml_viz_step.default_user_inputs

viz_inputs = default_viz_inputs()
viz_inputs['data_fbase'] = user_inputs.get('data_fbase', 'examples')
viz_inputs['data_subfolder'] = user_inputs.get('data_subfolder', '')
viz_inputs['metrics_summary_fname'] = f"{user_inputs['classification_or_regression']}_metrics_summary{user_inputs.get('csv_suffix', '')}.csv"
viz_inputs['model_name_list'] = ['best'] # ['xgboost', 'ridge']
viz_inputs['metric_col'] = ['test_spearman']  # string or list
viz_inputs['target_col_list'] = user_inputs.get('target_col', []) if isinstance(user_inputs.get('target_col', []), list) else [user_inputs.get('target_col')]
viz_inputs['split_type_list'] = []  # [] -> iterate all split types in summary CSV
viz_inputs['feature_label_list'] = list(user_inputs['feature_combinations_dict'].keys())
viz_inputs['show_model_type'] = False  # True -> marker shape based on model type
viz_inputs['save_figure'] = True
viz_inputs['show_figure'] = True
viz_inputs['figure_output_dir'] = ''  # default: summary csv folder
viz_inputs['figure_fname'] = ''       # default auto name
viz_inputs['x_limits'] = [0, None]  # e.g. [0, 10000]; None/[] uses matplotlib defaults
viz_inputs['y_limits'] = [0, 1]      # e.g. [0.0, 1.0]; None/[] uses matplotlib defaults

# Optional override:
viz_inputs['metrics_summary_csv_path'] = '/Users/charmainechia/Documents/projects/MUTAGENESIS-DATA-BENCHMARKS/ml/output/D7PM05_CLYGR_Somermeyer_2022/regression_metrics_summary_merged-0-1.csv'

viz_result = visualize_ml_results(viz_inputs)
pprint(viz_result)
